# Workbook 16 — SMS Customer Activation Analytics

## Customer Activation Intelligence System™

This workbook builds a customer activation intelligence layer for the Pizza House Grand Reopening SMS campaign.

The goal is to connect campaign delivery data, customer replies, verified engagement signals, and future marketing segmentation into one structured analytics system.

This workbook expands the Pizza House portfolio beyond restaurant revenue analysis by measuring how customer outreach performed after the move to **5050 Stockton Blvd**.

## 16.0 Business Context

Pizza House moved to a new location at **5050 Stockton Blvd** and launched a Grand Reopening SMS campaign using SimpleTexting.

Earlier Pizza House workbooks focused on orders, revenue, demand timing, customer records, and Tableau reporting. Workbook 16 focuses on customer activation.

The campaign data includes structured delivery reports, campaign summary screenshots, and unstructured reply screenshots. Because SimpleTexting does not provide a direct reply export, this workbook treats reply screenshots as a source of customer intelligence.

The final objective is to build a reusable marketing database that identifies delivered messages, verified customer responses, opt-outs, questions, invalid numbers, duplicate contacts, and future marketing-ready customers.

## 16.1 Customer Activation Framework

Workbook 16 follows a customer activation funnel:

```text
Customer List
    ↓
SMS Delivery
    ↓
Customer Replies
    ↓
Verified Customer Responses
    ↓
Customer Master Database
    ↓
Future Marketing Audience
```

The key business metric is **Verified Customer Responses**, not only exact "YES" replies.

Because coupon responses were delayed, some customers replied multiple times or used signals such as thumbs up, positive emojis, and HELP messages while waiting for the coupon. These are treated as verified engagement when the customer intent is clearly positive.

## 16.2 Setup — Imports and File Paths

This section defines the project folders used throughout Workbook 16.

The notebook follows the existing Pizza House repository structure and keeps source files, cleaned datasets, and final exports separated.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 50)

PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
CLEANED_DIR = DATA_DIR / "cleaned"
EXPORT_DIR = DATA_DIR / "exports"

SMS_DIR = RAW_DIR / "sms"
SC_DIR = SMS_DIR / "campaign_summaries"
DR_DIR = SMS_DIR / "delivery_reports"
RES_DIR = SMS_DIR / "replies"
IMPORTS_DIR = SMS_DIR / "imports"

folders = [
    RAW_DIR,
    CLEANED_DIR,
    EXPORT_DIR,
    SMS_DIR,
    SC_DIR,
    DR_DIR,
    RES_DIR,
    IMPORTS_DIR,
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("WB16 folders ready")
print("Campaign summaries:", SC_DIR)
print("Delivery reports:", DR_DIR)
print("Reply screenshots:", RES_DIR)
print("Imports:", IMPORTS_DIR)

WB16 folders ready
Campaign summaries: ../data/raw/sms/campaign_summaries
Delivery reports: ../data/raw/sms/delivery_reports
Reply screenshots: ../data/raw/sms/replies
Imports: ../data/raw/sms/imports


## 16.3 Source Data Inventory

This section confirms the files available for Workbook 16.

The source data is grouped into four categories:

- Campaign summary screenshots
- Delivery report CSV files
- Reply screenshots
- Future import or suppression files

This inventory step creates a quick audit trail before analysis begins.

In [2]:
sc_files = sorted(SC_DIR.glob("*"))
dr_files = sorted(DR_DIR.glob("*.csv"))
res_files = sorted(RES_DIR.glob("*"))
import_files = sorted(IMPORTS_DIR.glob("*"))

print("Campaign summary files:", len(sc_files))
for file in sc_files:
    print(" -", file.name)

print("\nDelivery report files:", len(dr_files))
for file in dr_files:
    print(" -", file.name)

print("\nReply screenshot files:", len(res_files))
for file in res_files:
    print(" -", file.name)

print("\nImport files:", len(import_files))
for file in import_files:
    print(" -", file.name)

Campaign summary files: 9
 - .ipynb_checkpoints
 - d1_a_sc.png
 - d1_b_sc.png
 - d2 _sc.png
 - d3_2_sc.png
 - d3_sc.png
 - d4_sc.png
 - d5_sc.png
 - d6_sc.png

Delivery report files: 8
 - d1_a_dr.csv
 - d1_b_dr.csv
 - d2_dr.csv
 - d3_2_dr.csv
 - d3_dr.csv
 - d4_dr.csv
 - d5_dr.csv
 - d6_dr.csv

Reply screenshot files: 20
 - d1_a_res.png
 - d1_b_res.png
 - d2_res_1.png
 - d2_res_2.png
 - d2_res_3.png
 - d2_res_4.png
 - d3_2_1_res.png
 - d3_2_2_res.png
 - d3_res_1.png
 - d3_res_2.png
 - d3_res_3.png
 - d3_res_4.png
 - d4_res_1.png
 - d4_res_2.png
 - d4_res_3.png
 - d5_res_1.png
 - d5_res_2.png
 - d5_res_3.png
 - d6_res_1.png
 - d6_res_2.png

Import files: 0


## 16.4 Campaign Timeline

The campaign was launched in multiple waves because SimpleTexting daily sending limits required the customer list to be split into smaller batches.

The final campaign structure included:

```text
D1_A
D1_B
D2
D3
D3_2
D4
D5
D6
```

The timeline is documented manually because it explains how the campaign evolved and why multiple campaign batches exist.

In [3]:
campaign_timeline = pd.DataFrame([
    {
        "campaign": "D1_A",
        "campaign_group": "Initial Test",
        "description": "First launch batch after Pizza House reopening message was prepared.",
        "notes": "Small batch used to begin campaign delivery."
    },
    {
        "campaign": "D1_B",
        "campaign_group": "Initial Test",
        "description": "Second launch batch sent after D1_A.",
        "notes": "Continued initial campaign rollout."
    },
    {
        "campaign": "D2",
        "campaign_group": "Scale Wave",
        "description": "Larger campaign wave after initial batches.",
        "notes": "Daily limits and platform behavior required campaign management."
    },
    {
        "campaign": "D3",
        "campaign_group": "Restart Wave",
        "description": "Campaign wave affected by campaign restart / overlap behavior.",
        "notes": "Tracked separately to preserve accurate source attribution."
    },
    {
        "campaign": "D3_2",
        "campaign_group": "System Constraint Wave",
        "description": "Additional D3-related campaign batch created because of platform constraints.",
        "notes": "Kept as its own campaign to avoid mixing source files."
    },
    {
        "campaign": "D4",
        "campaign_group": "Weekend Wave",
        "description": "Follow-up campaign batch after earlier waves.",
        "notes": "Part of continued customer activation rollout."
    },
    {
        "campaign": "D5",
        "campaign_group": "Final Wave",
        "description": "Final large customer activation wave.",
        "notes": "Sent after campaign structure was stabilized."
    },
    {
        "campaign": "D6",
        "campaign_group": "Final Wave",
        "description": "Final remaining customer activation wave.",
        "notes": "Completed remaining customer outreach."
    },
])

campaign_timeline.to_csv(CLEANED_DIR / "campaign_timeline.csv", index=False)

print("Exported:", CLEANED_DIR / "campaign_timeline.csv")

campaign_timeline

Exported: ../data/cleaned/campaign_timeline.csv


,campaign,campaign_group,description,notes
0,D1_A,Initial Test,First launch batch after Pizza House reopening...,Small batch used to begin campaign delivery.
1,D1_B,Initial Test,Second launch batch sent after D1_A.,Continued initial campaign rollout.
2,D2,Scale Wave,Larger campaign wave after initial batches.,Daily limits and platform behavior required ca...
3,D3,Restart Wave,Campaign wave affected by campaign restart / o...,Tracked separately to preserve accurate source...
4,D3_2,System Constraint Wave,Additional D3-related campaign batch created b...,Kept as its own campaign to avoid mixing sourc...
5,D4,Weekend Wave,Follow-up campaign batch after earlier waves.,Part of continued customer activation rollout.
6,D5,Final Wave,Final large customer activation wave.,Sent after campaign structure was stabilized.
7,D6,Final Wave,Final remaining customer activation wave.,Completed remaining customer outreach.


## 16.5 Campaign Summary Table

SimpleTexting campaign summary screenshots provide campaign-level information that is not available through delivery report exports.

This section creates a structured campaign summary table that can be updated from screenshots.

The table is intentionally manual because the screenshot metrics must be verified before they are used in executive reporting.

In [4]:
campaign_summary = pd.DataFrame([
    {
        "campaign": "D1_A",
        "campaign_group": "Initial Test",
        "contacts": 498,
        "send_date": "2026-06-08",
        "send_time": "15:30",
        "total_sent": np.nan,
        "delivered": np.nan,
        "failed": np.nan,
        "success_rate": np.nan,
        "notes": "Update delivery metrics from SimpleTexting campaign summary screenshot."
    },
    {
        "campaign": "D1_B",
        "campaign_group": "Initial Test",
        "contacts": 493,
        "send_date": "2026-06-08",
        "send_time": "15:45",
        "total_sent": np.nan,
        "delivered": np.nan,
        "failed": np.nan,
        "success_rate": np.nan,
        "notes": "Update delivery metrics from SimpleTexting campaign summary screenshot."
    },
    {
        "campaign": "D2",
        "campaign_group": "Scale Wave",
        "contacts": 2468,
        "send_date": "2026-06-09",
        "send_time": "various",
        "total_sent": np.nan,
        "delivered": np.nan,
        "failed": np.nan,
        "success_rate": np.nan,
        "notes": "Update delivery metrics from SimpleTexting campaign summary screenshot."
    },
    {
        "campaign": "D3",
        "campaign_group": "Restart Wave",
        "contacts": 996,
        "send_date": "2026-06-12",
        "send_time": "15:30",
        "total_sent": np.nan,
        "delivered": np.nan,
        "failed": np.nan,
        "success_rate": np.nan,
        "notes": "Update delivery metrics from SimpleTexting campaign summary screenshot."
    },
    {
        "campaign": "D3_2",
        "campaign_group": "System Constraint Wave",
        "contacts": np.nan,
        "send_date": "2026-06-10",
        "send_time": "various",
        "total_sent": np.nan,
        "delivered": np.nan,
        "failed": np.nan,
        "success_rate": np.nan,
        "notes": "Update contact and delivery metrics from screenshots."
    },
    {
        "campaign": "D4",
        "campaign_group": "Weekend Wave",
        "contacts": 1501,
        "send_date": "2026-06-14",
        "send_time": "14:00",
        "total_sent": np.nan,
        "delivered": np.nan,
        "failed": np.nan,
        "success_rate": np.nan,
        "notes": "Update delivery metrics from SimpleTexting campaign summary screenshot."
    },
    {
        "campaign": "D5",
        "campaign_group": "Final Wave",
        "contacts": 1509,
        "send_date": "2026-06-16",
        "send_time": "16:00",
        "total_sent": np.nan,
        "delivered": np.nan,
        "failed": np.nan,
        "success_rate": np.nan,
        "notes": "Update delivery metrics from SimpleTexting campaign summary screenshot."
    },
    {
        "campaign": "D6",
        "campaign_group": "Final Wave",
        "contacts": 1511,
        "send_date": "2026-06-17",
        "send_time": "16:00",
        "total_sent": np.nan,
        "delivered": np.nan,
        "failed": np.nan,
        "success_rate": np.nan,
        "notes": "Update delivery metrics from SimpleTexting campaign summary screenshot."
    },
])

campaign_summary["send_date"] = pd.to_datetime(campaign_summary["send_date"])

campaign_summary.to_csv(CLEANED_DIR / "campaign_summary.csv", index=False)
campaign_summary.to_csv(EXPORT_DIR / "campaign_summary.csv", index=False)

print("Exported:", CLEANED_DIR / "campaign_summary.csv")
print("Exported:", EXPORT_DIR / "campaign_summary.csv")

campaign_summary

Exported: ../data/cleaned/campaign_summary.csv
Exported: ../data/exports/campaign_summary.csv


,campaign,campaign_group,contacts,send_date,send_time,total_sent,delivered,failed,success_rate,notes
0,D1_A,Initial Test,498.0,2026-06-08,15:30,NaN,NaN,NaN,NaN,Update delivery metrics from SimpleTexting cam...
1,D1_B,Initial Test,493.0,2026-06-08,15:45,NaN,NaN,NaN,NaN,Update delivery metrics from SimpleTexting cam...
2,D2,Scale Wave,2468.0,2026-06-09,various,NaN,NaN,NaN,NaN,Update delivery metrics from SimpleTexting cam...
3,D3,Restart Wave,996.0,2026-06-12,15:30,NaN,NaN,NaN,NaN,Update delivery metrics from SimpleTexting cam...
4,D3_2,System Constraint Wave,NaN,2026-06-10,various,NaN,NaN,NaN,NaN,Update contact and delivery metrics from scree...
5,D4,Weekend Wave,1501.0,2026-06-14,14:00,NaN,NaN,NaN,NaN,Update delivery metrics from SimpleTexting cam...
6,D5,Final Wave,1509.0,2026-06-16,16:00,NaN,NaN,NaN,NaN,Update delivery metrics from SimpleTexting cam...
7,D6,Final Wave,1511.0,2026-06-17,16:00,NaN,NaN,NaN,NaN,Update delivery metrics from SimpleTexting cam...


## 16.6 Delivery Report Pipeline

Delivery reports are the structured SMS source data exported from SimpleTexting.

This section imports each delivery report, standardizes column names, attaches the campaign name from the file name, and appends all reports into one delivery master table.

In [5]:
delivery_frames = []

for path in sorted(DR_DIR.glob("*_dr.csv")):
    campaign = path.stem.replace("_dr", "").upper()

    temp = pd.read_csv(path)

    temp.columns = (
        temp.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )

    temp["campaign"] = campaign
    temp["source_file"] = path.name

    delivery_frames.append(temp)

    print(f"Loaded {campaign}: {len(temp):,} rows")

if delivery_frames:
    delivery_master = pd.concat(delivery_frames, ignore_index=True)
else:
    delivery_master = pd.DataFrame()
    print("No delivery report files found.")

print("Delivery master records:", len(delivery_master))

delivery_master.head()

Loaded D1_A: 497 rows
Loaded D1_B: 493 rows
Loaded D2: 2,975 rows
Loaded D3_2: 1,930 rows
Loaded D3: 866 rows
Loaded D4: 1,419 rows
Loaded D5: 1,391 rows
Loaded D6: 1,348 rows
Delivery master records: 10919


,phone_number,first_name,last_name,delivery_status,campaign,source_file
0,5304094821,22242,4220 Stockton Blvd,Opted Out,D1_A,d1_a_dr.csv
1,9164902587,12882,6 Lopis Crt,Delivered,D1_A,d1_a_dr.csv
2,9168852020,7970,5622 Florin Rd #70,Undelivered - Hard Bounce,D1_A,d1_a_dr.csv
3,9164902538,756,3610 25 Ave,Undelivered - Hard Bounce,D1_A,d1_a_dr.csv
4,2065811608,14226,4100 49th Ave Building 2 #15,Delivered,D1_A,d1_a_dr.csv


## 16.7 Delivery Data Cleaning

This section standardizes phone numbers and delivery statuses.

The cleaned fields make it possible to summarize delivery performance, identify invalid records, find duplicate contacts, and later merge delivery data with customer replies.

In [6]:
if not delivery_master.empty:
    phone_candidates = [col for col in delivery_master.columns if "phone" in col]
    phone_col = phone_candidates[0] if phone_candidates else None

    if phone_col:
        delivery_master["phone_clean"] = (
            delivery_master[phone_col]
            .astype(str)
            .str.replace(r"\D", "", regex=True)
        )
    else:
        delivery_master["phone_clean"] = np.nan

    status_candidates = [col for col in delivery_master.columns if "status" in col]
    status_col = status_candidates[0] if status_candidates else None

    if status_col:
        delivery_master["delivery_status_clean"] = (
            delivery_master[status_col]
            .astype(str)
            .str.strip()
            .str.lower()
        )
    else:
        delivery_master["delivery_status_clean"] = np.nan

    delivery_master.to_csv(CLEANED_DIR / "delivery_report_master.csv", index=False)

    print("Exported:", CLEANED_DIR / "delivery_report_master.csv")
    print("Phone column used:", phone_col)
    print("Status column used:", status_col)
else:
    print("Delivery master is empty. Add delivery report CSV files before running this section.")

delivery_master.head()

Exported: ../data/cleaned/delivery_report_master.csv
Phone column used: phone_number
Status column used: delivery_status


,phone_number,first_name,last_name,delivery_status,campaign,source_file,phone_clean,delivery_status_clean
0,5304094821,22242,4220 Stockton Blvd,Opted Out,D1_A,d1_a_dr.csv,5304094821,opted out
1,9164902587,12882,6 Lopis Crt,Delivered,D1_A,d1_a_dr.csv,9164902587,delivered
2,9168852020,7970,5622 Florin Rd #70,Undelivered - Hard Bounce,D1_A,d1_a_dr.csv,9168852020,undelivered - hard bounce
3,9164902538,756,3610 25 Ave,Undelivered - Hard Bounce,D1_A,d1_a_dr.csv,9164902538,undelivered - hard bounce
4,2065811608,14226,4100 49th Ave Building 2 #15,Delivered,D1_A,d1_a_dr.csv,2065811608,delivered


## 16.8 Delivery Status Summary

This section summarizes delivery status by campaign.

The output supports executive reporting and helps identify campaign batches with failed, invalid, or undelivered numbers.

In [7]:
if not delivery_master.empty:
    delivery_status_summary = (
        delivery_master
        .groupby(["campaign", "delivery_status_clean"], dropna=False)
        .size()
        .reset_index(name="record_count")
        .sort_values(["campaign", "record_count"], ascending=[True, False])
    )

    delivery_status_summary.to_csv(
        CLEANED_DIR / "delivery_status_summary.csv",
        index=False
    )

    print("Exported:", CLEANED_DIR / "delivery_status_summary.csv")
else:
    delivery_status_summary = pd.DataFrame(
        columns=["campaign", "delivery_status_clean", "record_count"]
    )

    print("Delivery status summary not created because delivery master is empty.")

delivery_status_summary

Exported: ../data/cleaned/delivery_status_summary.csv


,campaign,delivery_status_clean,record_count
0,D1_A,delivered,323
2,D1_A,undelivered - hard bounce,92
3,D1_A,undelivered - soft bounce,43
1,D1_A,opted out,39
4,D1_B,delivered,342
6,D1_B,undelivered - hard bounce,95
5,D1_B,opted out,29
7,D1_B,undelivered - soft bounce,27
8,D2,delivered,2005
10,D2,undelivered - hard bounce,546


## 16.9 Reply Tracker Batch Inventory

Reply tracker CSV files are created one campaign at a time from the screenshot extraction workflow.

Each campaign is stored independently in the `data/cleaned/replies/` folder. This allows every batch to be reviewed and cleaned before being combined into a master reply tracker.

This section verifies which reply tracker batches currently exist.


In [8]:
REPLIES_DIR = CLEANED_DIR / "replies"

REPLIES_DIR.mkdir(parents=True, exist_ok=True)

reply_batch_files = sorted(REPLIES_DIR.glob("*_reply_tracker.csv"))

print(f"Reply tracker batches found: {len(reply_batch_files)}")

for file in reply_batch_files:
    print(f" - {file.name}")

Reply tracker batches found: 1
 - d1_reply_tracker.csv


## 16.10 Verified Customer Response Framework

Workbook 16 measures **Verified Customer Responses** instead of only exact "YES" replies.

This business rule reflects how the campaign actually performed. Because coupon responses were delayed, some customers sent additional confirmations, emoji responses, or HELP messages while waiting for the coupon.

A verified customer response includes positive intent to redeem or engage with the promotion.

Examples include:

```text
YES
Yes
Y
Yep
Yeah
Sí
OK
Okay
👍
Positive emoji responses
HELP related to delayed coupon response
Additional positive follow-up confirmations
```

Multiple positive replies from the same customer should count as one verified customer response unless the conversation clearly shows a different intent.

In [9]:
verified_response_terms = [
    "yes",
    "y",
    "yep",
    "yeah",
    "si",
    "sí",
    "ok",
    "okay",
    "👍",
    "❤️",
    "😊",
    "😁",
]

question_terms = [
    "address",
    "cross",
    "where",
    "hours",
    "open",
    "location",
    "when",
    "what time",
]

negative_terms = [
    "no",
    "nah",
    "nope",
    "don't care",
    "dont care",
]

wrong_number_terms = [
    "wrong",
    "remove",
    "not me",
]

print("Verified response terms:", len(verified_response_terms))
print("Question terms:", len(question_terms))
print("Negative terms:", len(negative_terms))
print("Wrong number terms:", len(wrong_number_terms))

Verified response terms: 12
Question terms: 8
Negative terms: 5
Wrong number terms: 3


## 16.11 Reply Classification Function

This function applies the Workbook 16 business rules to customer reply text.

The function is intentionally readable and journal-friendly. It prioritizes business interpretation over overly complex text processing.

In [10]:
def classify_reply(text):
    if pd.isna(text):
        return "UNKNOWN"

    value = str(text).strip().lower()

    if value == "":
        return "UNKNOWN"

    if "stop" in value:
        return "STOP"

    if any(term in value for term in wrong_number_terms):
        return "WRONG_NUMBER"

    if value == "help" or "help" in value:
        return "VERIFIED_RESPONSE"

    if (
        value in verified_response_terms
        or value.startswith("yes")
        or any(term in value for term in ["👍", "❤️", "😊", "😁"])
    ):
        return "VERIFIED_RESPONSE"

    if any(term in value for term in question_terms):
        return "QUESTION"

    if value in negative_terms or any(term in value for term in negative_terms):
        return "NEGATIVE"

    return "OTHER"


example_replies = [
    "YES",
    "👍",
    "HELP",
    "What's the cross street?",
    "STOP",
    "wrong number",
    "No",
    "Thanks!",
]

reply_examples = pd.DataFrame({
    "reply_text": example_replies,
    "reply_category": [classify_reply(reply) for reply in example_replies],
})

reply_examples

,reply_text,reply_category
0,YES,VERIFIED_RESPONSE
1,👍,VERIFIED_RESPONSE
2,HELP,VERIFIED_RESPONSE
3,What's the cross street?,QUESTION
4,STOP,STOP
5,wrong number,WRONG_NUMBER
6,No,NEGATIVE
7,Thanks!,OTHER


## 16.12 Reply Tracker Schema

The reply tracker is the structured dataset used to store customer responses extracted from reply screenshots.

The workflow is now OCR-assisted instead of fully manual. Screenshot text will be extracted, classified, reviewed, and then stored in this table.

This prevents the project from becoming manual data entry and turns the reply screenshots into an analytics pipeline.

In [11]:
reply_tracker = pd.DataFrame(columns=[
    "campaign",
    "source_file",
    "contact_display",
    "phone_clean",
    "reply_text",
    "reply_category",
    "coupon_sent",
    "follow_up_needed",
    "verified_response_flag",
    "notes",
])

reply_tracker.to_csv(
    CLEANED_DIR / "reply_tracker.csv",
    index=False
)

print("Exported:", CLEANED_DIR / "reply_tracker.csv")

reply_tracker

Exported: ../data/cleaned/reply_tracker.csv


,campaign,source_file,contact_display,phone_clean,reply_text,reply_category,coupon_sent,follow_up_needed,verified_response_flag,notes


## 16.13 OCR-Assisted Reply Extraction Plan

This section documents the planned extraction approach for reply screenshots.

The raw screenshots will be processed into a structured reply tracker using this workflow:

```text
Reply Screenshots
    ↓
OCR Text Extraction
    ↓
Conversation Parsing
    ↓
Reply Classification
    ↓
Human Quality Review
    ↓
reply_tracker.csv
```

The goal is not to manually type every customer response. The goal is to use automation to create a first-pass reply dataset, then manually review only ambiguous or low-confidence records.

In [12]:
ocr_extraction_plan = pd.DataFrame([
    {
        "step": 1,
        "process": "Load reply screenshots",
        "output": "Screenshot inventory"
    },
    {
        "step": 2,
        "process": "Extract visible text from each screenshot",
        "output": "Raw OCR text"
    },
    {
        "step": 3,
        "process": "Identify contact display values and reply text",
        "output": "Draft reply records"
    },
    {
        "step": 4,
        "process": "Apply Workbook 16 reply classification rules",
        "output": "Draft reply categories"
    },
    {
        "step": 5,
        "process": "Review ambiguous records manually",
        "output": "Clean reply tracker"
    },
])

ocr_extraction_plan

,step,process,output
0,1,Load reply screenshots,Screenshot inventory
1,2,Extract visible text from each screenshot,Raw OCR text
2,3,Identify contact display values and reply text,Draft reply records
3,4,Apply Workbook 16 reply classification rules,Draft reply categories
4,5,Review ambiguous records manually,Clean reply tracker


## 16.14 Data Quality Outputs

Delivery reports can also be used to identify invalid numbers and duplicate phone records.

These outputs are operationally valuable because they improve the quality of future marketing campaigns and reduce wasted sends.

In [13]:
if not delivery_master.empty:
    invalid_keywords = ["invalid", "failed", "undelivered", "error"]

    invalid_numbers = delivery_master[
        delivery_master["delivery_status_clean"]
        .astype(str)
        .str.contains("|".join(invalid_keywords), na=False)
    ].copy()

    duplicate_phones = (
        delivery_master[delivery_master["phone_clean"].notna()]
        .groupby("phone_clean")
        .size()
        .reset_index(name="record_count")
        .query("record_count > 1")
        .sort_values("record_count", ascending=False)
    )

    invalid_numbers.to_csv(CLEANED_DIR / "invalid_numbers.csv", index=False)
    duplicate_phones.to_csv(CLEANED_DIR / "duplicate_phones.csv", index=False)

    print("Exported:", CLEANED_DIR / "invalid_numbers.csv")
    print("Exported:", CLEANED_DIR / "duplicate_phones.csv")
    print("Invalid number records:", len(invalid_numbers))
    print("Duplicate phone records:", len(duplicate_phones))
else:
    invalid_numbers = pd.DataFrame()
    duplicate_phones = pd.DataFrame()

    print("Data quality outputs not created because delivery master is empty.")

Exported: ../data/cleaned/invalid_numbers.csv
Exported: ../data/cleaned/duplicate_phones.csv
Invalid number records: 2561
Duplicate phone records: 1524


## 16.15 Initial Activation Funnel

This section creates the first version of the SMS activation funnel using available delivery data.

Reply and verified response metrics will be added after OCR-assisted reply extraction and quality review are complete.

In [14]:
if not delivery_master.empty:
    total_delivery_records = len(delivery_master)
    unique_phone_records = delivery_master["phone_clean"].nunique()
    invalid_record_count = len(invalid_numbers)

    activation_funnel = pd.DataFrame([
        {
            "stage": "Delivery Report Records",
            "count": total_delivery_records,
        },
        {
            "stage": "Unique Phone Numbers",
            "count": unique_phone_records,
        },
        {
            "stage": "Invalid / Failed Records",
            "count": invalid_record_count,
        },
    ])
else:
    activation_funnel = pd.DataFrame([
        {
            "stage": "Delivery Report Records",
            "count": 0,
        },
        {
            "stage": "Unique Phone Numbers",
            "count": 0,
        },
        {
            "stage": "Invalid / Failed Records",
            "count": 0,
        },
    ])

activation_funnel.to_csv(
    EXPORT_DIR / "sms_activation_funnel.csv",
    index=False
)

print("Exported:", EXPORT_DIR / "sms_activation_funnel.csv")

activation_funnel

Exported: ../data/exports/sms_activation_funnel.csv


,stage,count
0,Delivery Report Records,10919
1,Unique Phone Numbers,9377
2,Invalid / Failed Records,2561


## 16.16 Planned Final Outputs

The final version of Workbook 16 will produce a customer activation intelligence layer with operational and executive outputs.

Planned cleaned outputs:

```text
campaign_timeline.csv
campaign_summary.csv
delivery_report_master.csv
delivery_status_summary.csv
reply_screenshot_inventory.csv
reply_tracker.csv
invalid_numbers.csv
duplicate_phones.csv
customer_master.csv
verified_customer_responses.csv
opt_out_customers.csv
customer_questions.csv
wrong_numbers.csv
future_marketing_database.csv
```

Planned export outputs:

```text
sms_activation_funnel.csv
sms_activation_funnel_final.csv
tableau_sms_customer_activation.csv
```

## 16.17 Next Steps

1. Run the starter workbook to verify folder paths and source file inventory.
2. Confirm delivery report imports and delivery status cleaning.
3. Use OCR-assisted extraction to create a first-pass reply tracker from screenshots.
4. Apply the Verified Customer Response framework.
5. Review ambiguous reply records manually.
6. Merge delivery and reply data into `customer_master.csv`.
7. Export action lists for verified responses, opt-outs, questions, wrong numbers, and future marketing.
8. Build the Tableau-ready SMS Customer Activation dataset.
9. Add executive findings and update the GitHub README.

## 16.8 Delivery Status Summary

Delivery status provides the first measurement of SMS campaign execution.

This section summarizes delivery outcomes by campaign so each batch can be evaluated before customer replies are analyzed.

These metrics establish which customers were successfully reached, which numbers failed, and which records should be excluded or reviewed before building the final customer activation layer. 

In [15]:
delivery_status_summary = (
    delivery_master
    .groupby(
        ["campaign", "delivery_status_clean"],
        dropna=False
    )
    .size()
    .reset_index(name="record_count")
    .sort_values(
        ["campaign", "record_count"],
        ascending=[True, False]
    )
)

delivery_status_summary.to_csv(
    CLEANED_DIR / "delivery_status_summary.csv",
    index=False
)

print("Exported:", CLEANED_DIR / "delivery_status_summary.csv")

delivery_status_summary

Exported: ../data/cleaned/delivery_status_summary.csv


,campaign,delivery_status_clean,record_count
0,D1_A,delivered,323
2,D1_A,undelivered - hard bounce,92
3,D1_A,undelivered - soft bounce,43
1,D1_A,opted out,39
4,D1_B,delivered,342
6,D1_B,undelivered - hard bounce,95
5,D1_B,opted out,29
7,D1_B,undelivered - soft bounce,27
8,D2,delivered,2005
10,D2,undelivered - hard bounce,546


## 16.9 Reply Tracker Batch Import

Reply tracker CSV files are generated one campaign at a time through the screenshot extraction workflow.

Each campaign batch is stored independently in the `data/cleaned/replies/` folder, allowing reply data to be reviewed and cleaned before being combined into a master reply tracker.

This section imports all available reply tracker batches and verifies the imported schema before downstream cleaning.

In [28]:
REPLIES_DIR = CLEANED_DIR / "replies"

REPLIES_DIR.mkdir(parents=True, exist_ok=True)

reply_batch_files = sorted(REPLIES_DIR.glob("*_reply_tracker.csv"))

print(f"Reply tracker batches found: {len(reply_batch_files)}")

for file in reply_batch_files:
    print(f" - {file.name}")

if reply_batch_files:
    reply_tracker = pd.read_csv(reply_batch_files[0])

    print("\nImported:", reply_batch_files[0].name)
    print(f"Records: {len(reply_tracker):,}")

    print("\nColumns:")
    print(reply_tracker.columns.tolist())

    display(reply_tracker.head())
else:
    raise FileNotFoundError(
        "No reply tracker batches were found in data/cleaned/replies/."
    )

Reply tracker batches found: 1
 - d1_reply_tracker.csv

Imported: d1_reply_tracker.csv
Records: 44

Columns:
['campaign', 'source_file', 'address_raw', 'phone_clean', 'reply_text', 'engagement_status', 'coupon_sent', 'follow_up_needed', 'notes']


,campaign,source_file,address_raw,phone_clean,reply_text,engagement_status,coupon_sent,follow_up_needed,notes
0,D1_A,d1_a_res.png,a 130 7350 Alden Way 13d8,NaN,| think you moved 1 mile out my r...,YES,Yes,No,NaN
1,D1_A,d1_a_res.png,4080 6425 Somis 13d @1,NaN,Yes,YES,Yes,No,NaN
2,D1_A,d1_a_res.png,a 13755 2174 55 Ave 13d @1,NaN,YES,YES,Yes,No,NaN
3,D1_A,d1_a_res.png,10189 5505 Sky Parkway ... 13d & @1,NaN,Yes,YES,Yes,No,NaN
4,D1_A,d1_a_res.png,20945 6618 Sunriver Dr 13d8 @1,NaN,Yes,YES,Yes,No,NaN


## 16.10 Initial Reply Tracker Cleaning

Reply tracker batches extracted from screenshot conversations often contain OCR artifacts, inconsistent spacing, and non-address text that can interfere with downstream customer matching.

This section performs the initial cleaning of reply tracker records by standardizing text fields and removing common OCR artifacts. Address parsing and customer matching are performed in later sections.

In [29]:
# Standardize column names
reply_tracker.columns = (
    reply_tracker.columns
    .str.strip()
    .str.lower()
)

# Trim whitespace from all text columns
text_columns = reply_tracker.select_dtypes(include="object").columns

reply_tracker[text_columns] = (
    reply_tracker[text_columns]
    .apply(lambda column: column.str.strip())
)

# Clean extracted address text
reply_tracker["address_clean"] = (
    reply_tracker["address_raw"]
        .str.replace(r"\b\d+d\d*\b", "", regex=True)   # 13d, 13d8, 12d, etc.
        .str.replace(r"@\d+", "", regex=True)          # @1, @2
        .str.replace(r"&\s*@\d+", "", regex=True)      # & @1
        .str.replace(r"^\s*a\s+", "", regex=True)      # leading "a "
        .str.replace(r"\.{2,}", "", regex=True)        # ...
        .str.replace(r"\s+", " ", regex=True)          # extra spaces
        .str.strip()
)

# Clean reply text spacing
reply_tracker["reply_text"] = (
    reply_tracker["reply_text"]
        .fillna("")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
)

print(f"Reply tracker records: {len(reply_tracker):,}")

display(
    reply_tracker[
        [
            "campaign",
            "address_raw",
            "address_clean",
            "reply_text"
        ]
    ].head(15)
)

Reply tracker records: 44


,campaign,address_raw,address_clean,reply_text
0,D1_A,a 130 7350 Alden Way 13d8,130 7350 Alden Way,| think you moved 1 mile out my r...
1,D1_A,4080 6425 Somis 13d @1,4080 6425 Somis,Yes
2,D1_A,a 13755 2174 55 Ave 13d @1,13755 2174 55 Ave,YES
3,D1_A,10189 5505 Sky Parkway ... 13d & @1,10189 5505 Sky Parkway &,Yes
4,D1_A,20945 6618 Sunriver Dr 13d8 @1,20945 6618 Sunriver Dr,Yes
5,D1_A,867 5642 Odea DrApt3 13d @1,867 5642 Odea DrApt3,Yes
6,D1_A,7200 5923 69th St 13d @1,7200 5923 69th St,Yes
7,D1_A,6246 6311 Sampson Blvd... 13d & @1,6246 6311 Sampson Blvd &,Yes
8,D1_A,4018 7831 Lemon Hill Ave 13d & @1,4018 7831 Lemon Hill Ave &,Yes!
9,D1_A,a 21209 7732 40 Ave 13d @1,21209 7732 40 Ave,YES


## 16.11 Address Parsing

Reply tracker addresses exported from the SMS platform begin with an internal conversation identifier followed by the customer's address.

This section removes the conversation identifier, parses the remaining customer address into standardized components, and creates a reusable address join key for downstream customer matching.

The parser preserves numeric street names (e.g. "55 Ave"), supports addresses without street suffixes, and separates apartment or unit information when available.

In [62]:
import re

# Remove previously generated parsing columns if this cell is rerun
parsed_columns = [
    "street_number",
    "street_name",
    "street_suffix",
    "unit",
    "address_join_key"
]

reply_tracker = reply_tracker.drop(
    columns=[c for c in parsed_columns if c in reply_tracker.columns],
    errors="ignore"
)

STREET_SUFFIXES = {
    "ST", "AVE", "DR", "RD",
    "BLVD", "CT", "CIR",
    "WAY", "LN", "PL",
    "PKWY", "TER", "TRL", "HWY"
}

UNIT_PREFIXES = (
    "APT",
    "UNIT",
    "#",
    "ROOM",
    "RM",
    "STE",
    "SUITE"
)

ocr_corrections = {
    "BIV": "BLVD",
    "STOCKTEN": "STOCKTON",
    "PK": "PKWY"
}

parsed_rows = []

for address in reply_tracker["address_clean"]:

    street_number = None
    street_name = None
    street_suffix = None
    unit = None
    address_join_key = None

    if pd.notna(address):

        text = str(address).upper().strip()
        text = re.sub(r"\b\d+D\d*\b", "", text)
        text = re.sub(r"@\d+", "", text)
        text = text.replace("&", " ")
        text = re.sub(r"\s+", " ", text).strip()

        for bad, good in ocr_corrections.items():
            text = re.sub(rf"\b{bad}\b", good, text)

        text = re.sub(
            r"(ST|AVE|DR|RD|BLVD|CT|CIR|WAY|LN|PL|PKWY)(APT|UNIT|ROOM|RM|STE|SUITE)",
            r"\1 \2",
            text
        )

        tokens = text.split()

        # Remove conversation identifier
        if tokens and tokens[0].isdigit():
            tokens = tokens[1:]

        # Customer street number
        if tokens and tokens[0].isdigit():

            street_number = tokens.pop(0)

            # Extract apartment / unit information
            for i, token in enumerate(tokens):

                token_upper = token.upper()

                if (
                    token_upper.startswith(UNIT_PREFIXES)
                    or "#" in token_upper
                ):
                    unit = " ".join(tokens[i:])
                    tokens = tokens[:i]
                    break

            # Extract street suffix
            for i, token in enumerate(tokens):

                token_upper = token.upper()

                if token_upper in STREET_SUFFIXES:
                    street_suffix = token_upper
                    tokens = tokens[:i]
                    break

            street_name = " ".join(tokens)
            street_name = re.sub(r"\s+", " ", street_name).strip()

            join_parts = [street_number]

            if street_name:
                join_parts.append(
                    street_name.replace(" ", "_")
                )

            if street_suffix:
                join_parts.append(street_suffix)

            address_join_key = "_".join(join_parts)

    parsed_rows.append({
        "street_number": street_number,
        "street_name": street_name,
        "street_suffix": street_suffix,
        "unit": unit,
        "address_join_key": address_join_key
    })

parsed_rows = pd.DataFrame(parsed_rows)

reply_tracker = pd.concat(
    [reply_tracker, parsed_rows],
    axis=1
)

print(f"Addresses parsed: {len(reply_tracker):,}")

display(
    reply_tracker[
        [
            "address_clean",
            "street_number",
            "street_name",
            "street_suffix",
            "unit",
            "address_join_key"
        ]
    ].head(25)
)

Addresses parsed: 44


,address_clean,street_number,street_name,street_suffix,unit,address_join_key
0,130 7350 Alden Way,7350,ALDEN,WAY,None,7350_ALDEN_WAY
1,4080 6425 Somis,6425,SOMIS,None,None,6425_SOMIS
2,13755 2174 55 Ave,2174,55,AVE,None,2174_55_AVE
3,10189 5505 Sky Parkway &,5505,SKY PARKWAY,None,None,5505_SKY_PARKWAY
4,20945 6618 Sunriver Dr,6618,SUNRIVER,DR,None,6618_SUNRIVER_DR
5,867 5642 Odea DrApt3,5642,ODEA,DR,APT3,5642_ODEA_DR
6,7200 5923 69th St,5923,69TH,ST,None,5923_69TH_ST
7,6246 6311 Sampson Blvd &,6311,SAMPSON,BLVD,None,6311_SAMPSON_BLVD
8,4018 7831 Lemon Hill Ave &,7831,LEMON HILL,AVE,None,7831_LEMON_HILL_AVE
9,21209 7732 40 Ave,7732,40,AVE,None,7732_40_AVE


## 16.12 Customer Master Import

The cleaned customer list serves as the master dataset for Workbook 16.

This dataset contains the verified customer phone numbers and mailing addresses used to connect reply tracker records with SMS delivery reports. All downstream joins are performed through this master customer table.

In [55]:
CUSTOMER_FILE = CLEANED_DIR / "campaign_sms_test_list.csv"

customer_master = pd.read_csv(CUSTOMER_FILE)

print(f"Customer records: {len(customer_master):,}")
print(f"Columns: {len(customer_master.columns)}")

display(customer_master.head())

Customer records: 9,557
Columns: 3


,customer_id_clean,Phone Number,Address Line 1
0,1,9162129352,6710 37 Ave
1,4,9162301496,6425 Somis Way
2,5,9163292467,6100 48th Ave #2202
3,6,2792720793,3826 25th Ave
4,7,9162065582,7101 Gerber Rd #216 #8106


## 16.13 Customer Address Parsing

Customer mailing addresses are standardized using the same parsing approach applied to the reply tracker.

Each address is separated into its street number, street name, street suffix, apartment or unit information, and a standardized address join key. Using identical parsing logic across both datasets allows reliable address-based matching in subsequent sections.

In [56]:
import re

# Remove previously generated parsing columns if this cell is rerun
parsed_columns = [
    "address_clean",
    "street_number",
    "street_name",
    "street_suffix",
    "unit",
    "address_join_key"
]

customer_master = customer_master.drop(
    columns=[c for c in parsed_columns if c in customer_master.columns],
    errors="ignore"
)

STREET_SUFFIXES = {
    "ST", "AVE", "DR", "RD",
    "BLVD", "CT", "CIR",
    "WAY", "LN", "PL",
    "PKWY", "TER", "TRL", "HWY"
}

UNIT_PREFIXES = (
    "APT",
    "UNIT",
    "#",
    "ROOM",
    "RM",
    "STE",
    "SUITE"
)

parsed_rows = []

for address in customer_master["Address Line 1"]:

    address_clean = None
    street_number = None
    street_name = None
    street_suffix = None
    unit = None
    address_join_key = None

    if pd.notna(address):

        text = str(address).upper().strip()
        text = re.sub(r"\s+", " ", text)

        address_clean = text

        tokens = text.split()

        if tokens and tokens[0].isdigit():

            street_number = tokens.pop(0)

            # Extract apartment / unit information
            for i, token in enumerate(tokens):

                token_upper = token.upper()

                if (
                    token_upper.startswith(UNIT_PREFIXES)
                    or "#" in token_upper
                ):
                    unit = " ".join(tokens[i:])
                    tokens = tokens[:i]
                    break

            # Extract street suffix
            for i, token in enumerate(tokens):

                token_upper = token.upper()

                if token_upper in STREET_SUFFIXES:
                    street_suffix = token_upper
                    tokens = tokens[:i]
                    break

            street_name = " ".join(tokens)

            join_parts = [street_number]

            if street_name:
                join_parts.append(
                    street_name.replace(" ", "_")
                )

            if street_suffix:
                join_parts.append(street_suffix)

            address_join_key = "_".join(join_parts)

    parsed_rows.append({
        "address_clean": address_clean,
        "street_number": street_number,
        "street_name": street_name,
        "street_suffix": street_suffix,
        "unit": unit,
        "address_join_key": address_join_key
    })

parsed_rows = pd.DataFrame(parsed_rows)

customer_master = pd.concat(
    [customer_master, parsed_rows],
    axis=1
)

print(f"Customer addresses parsed: {len(customer_master):,}")

display(
    customer_master[
        [
            "Address Line 1",
            "address_clean",
            "street_number",
            "street_name",
            "street_suffix",
            "unit",
            "address_join_key"
        ]
    ].head(20)
)

Customer addresses parsed: 9,557


,Address Line 1,address_clean,street_number,street_name,street_suffix,unit,address_join_key
0,6710 37 Ave,6710 37 AVE,6710,37,AVE,None,6710_37_AVE
1,6425 Somis Way,6425 SOMIS WAY,6425,SOMIS,WAY,None,6425_SOMIS_WAY
2,6100 48th Ave #2202,6100 48TH AVE #2202,6100,48TH,AVE,#2202,6100_48TH_AVE
3,3826 25th Ave,3826 25TH AVE,3826,25TH,AVE,None,3826_25TH_AVE
4,7101 Gerber Rd #216 #8106,7101 GERBER RD #216 #8106,7101,GERBER,RD,#216 #8106,7101_GERBER_RD
5,5677 23rd St,5677 23RD ST,5677,23RD,ST,None,5677_23RD_ST
6,6267 Mlk Jr Blvd Apt 122,6267 MLK JR BLVD APT 122,6267,MLK JR,BLVD,APT 122,6267_MLK_JR_BLVD
7,7809 36th Ave,7809 36TH AVE,7809,36TH,AVE,None,7809_36TH_AVE
8,4709 47th St,4709 47TH ST,4709,47TH,ST,None,4709_47TH_ST
9,6448 Stockton Blvd Room 27,6448 STOCKTON BLVD ROOM 27,6448,STOCKTON,BLVD,ROOM 27,6448_STOCKTON_BLVD


## 16.14 Reply-to-Customer Match

Reply tracker records are matched back to the SMS campaign customer list using the standardized `address_join_key`.

Because some addresses contain multiple customer records, apartments, or phone numbers, the customer table is first reduced to one representative record per address join key. This prevents one reply from duplicating across multiple customer records during the merge.

The result is a one-row-per-reply dataset enriched with customer identifiers and phone numbers where a match is available.

In [63]:
customer_lookup = (
    customer_master[
        [
            "customer_id_clean",
            "Phone Number",
            "Address Line 1",
            "address_join_key"
        ]
    ]
    .dropna(subset=["address_join_key"])
    .sort_values("customer_id_clean")
    .drop_duplicates(
        subset="address_join_key",
        keep="first"
    )
)

reply_customer_match = reply_tracker.merge(
    customer_lookup,
    on="address_join_key",
    how="left"
)

matched_replies = reply_customer_match["Phone Number"].notna().sum()
total_replies = len(reply_customer_match)

match_rate = matched_replies / total_replies

print(f"Reply records: {total_replies:,}")
print(f"Matched replies: {matched_replies:,}")
print(f"Match rate: {match_rate:.1%}")

display(
    reply_customer_match[
        [
            "campaign",
            "address_clean",
            "address_join_key",
            "Phone Number",
            "Address Line 1",
            "reply_text",
            "engagement_status",
            "coupon_sent"
        ]
    ].head(25)
)

Reply records: 44
Matched replies: 26
Match rate: 59.1%


,campaign,address_clean,address_join_key,Phone Number,Address Line 1,reply_text,engagement_status,coupon_sent
0,D1_A,130 7350 Alden Way,7350_ALDEN_WAY,9169955697,7350 Alden Way,| think you moved 1 mile out my r...,YES,Yes
1,D1_A,4080 6425 Somis,6425_SOMIS,9163460667,6425 Somis,Yes,YES,Yes
2,D1_A,13755 2174 55 Ave,2174_55_AVE,9167935450,2174 55 Ave,YES,YES,Yes
3,D1_A,10189 5505 Sky Parkway &,5505_SKY_PARKWAY,9164779101,5505 Sky Parkway Apt2,Yes,YES,Yes
4,D1_A,20945 6618 Sunriver Dr,6618_SUNRIVER_DR,9167104841,6618 Sunriver Dr,Yes,YES,Yes
5,D1_A,867 5642 Odea DrApt3,5642_ODEA_DR,9166708903,5642 Odea Dr Apt 3,Yes,YES,Yes
6,D1_A,7200 5923 69th St,5923_69TH_ST,9168770044,5923 69th St,Yes,YES,Yes
7,D1_A,6246 6311 Sampson Blvd &,6311_SAMPSON_BLVD,2792712197,6311 Sampson Blvd #54,Yes,YES,Yes
8,D1_A,4018 7831 Lemon Hill Ave &,7831_LEMON_HILL_AVE,9168867741,7831 Lemon Hill Ave,Yes!,YES,Yes
9,D1_A,21209 7732 40 Ave,7732_40_AVE,9167189428,7732 40 Ave,YES,YES,Yes


## 16.15 Unmatched Reply Review

After reply records are matched to the campaign customer list, unmatched records are reviewed separately.

Most unmatched records are caused by OCR extraction issues, truncated addresses, or inconsistent street spelling. This review step creates a focused QA table so only unmatched reply records need correction before final customer matching.

In [65]:
unmatched_replies = reply_customer_match[
    reply_customer_match["Phone Number"].isna()
].copy()

print(f"Unmatched replies: {len(unmatched_replies):,}")

display(
    unmatched_replies[
        [
            "campaign",
            "address_clean",
            "address_join_key",
            "reply_text",
            "engagement_status",
            "coupon_sent"
        ]
    ]
)

Unmatched replies: 18


,campaign,address_clean,address_join_key,reply_text,engagement_status,coupon_sent
16,D1_A,10593 6901 Casa Del Est 8,6901_CASA_DEL_EST_8,Yes,YES,Yes
23,D1_B,8007 Zlata Ct,None,Yes,YES,Yes
26,D1_B,22796 4622 FlorinR ‘245,4622_FLORINR_‘245,YES,YES,Yes
27,D1_B,3384 5200 Stockton. ‘25,5200_STOCKTON._‘25,Yes,YES,Yes
28,D1_B,T1608 4317 54th St ‘ids,None,P,YES,Yes
30,D1_B,4567746750 Ave ‘20501,None,Yes,YES,Yes
31,D1_B,46333932 33thSt ‘lds!,None,Yes,YES,Yes
32,D1_B,682 7501StstAve ‘ el,None,Yes,YES,Yes
33,D1_B,637 Fairgrounds Dr #3,None,Yes,YES,Yes
34,D1_B,5841 25 Ave,25_AVE,Yes,YES,Yes


## 16.16 Manual Address Corrections

Unmatched reply records are reviewed against the original reply screenshots.

When OCR produces incomplete or incorrect addresses, a small correction table is created and applied before re-running the customer match. This keeps the parser general while allowing campaign-level QA corrections to remain transparent and reproducible.

In [66]:
address_corrections = pd.DataFrame([
    {"address_clean": "4102 8007 Ziata Ct 134 8", "corrected_address_clean": "8007 Zlata Ct"},
    {"address_clean": "22796 4622 FlorinR '245", "corrected_address_clean": "4622 Florin Rd #6"},
    {"address_clean": "3384 5200 Stockton. '25", "corrected_address_clean": "5200 Stockton Blvd #130"},
    {"address_clean": "T1608 4317 54th St 'ids", "corrected_address_clean": "4317 54th St"},
    {"address_clean": "4567746750 Ave '20501", "corrected_address_clean": "7467 50 Ave"},
    {"address_clean": "46333932 33thSt '/ds!", "corrected_address_clean": "3932 39th St"},
    {"address_clean": "682 7501StstAve ' el", "corrected_address_clean": "7501 51st Ave"},
    {"address_clean": "22956 637 Fairgroun '24 5", "corrected_address_clean": "637 Fairgrounds Dr #3"},
    {"address_clean": "12155 584125 Ave [ids", "corrected_address_clean": "5841 25 Ave"},
    {"address_clean": "13530 3908 42nd Av '24 5", "corrected_address_clean": "3908 42nd Ave #E"},
    {"address_clean": "19332 4325 35Ave 'ids 01", "corrected_address_clean": "4325 35 Ave"},
    {"address_clean": "21043 6000 Lemon. '245", "corrected_address_clean": "6000 Lemon Hill Ave"},
    {"address_clean": "a@ 1119 6504 18 Ave 'deel", "corrected_address_clean": "6504 18 Ave"},
    {"address_clean": "6957 4113 Ghaddi Or._ '245", "corrected_address_clean": "4113 Ghaddi Dr #3"},
    {"address_clean": "20771 4971 Concord 245", "corrected_address_clean": "4971 Concord Rd"},
    {"address_clean": "328 7931 Capistrano 245", "corrected_address_clean": "7931 Capistrano Way"},
    {"address_clean": "13779 482061St 'ids el", "corrected_address_clean": "4820 61st St"},
])

address_corrections.to_csv(
    REPLIES_DIR / "d1_address_corrections.csv",
    index=False
)

print("Address corrections:", len(address_corrections))
print("Exported:", REPLIES_DIR / "d1_address_corrections.csv")

address_corrections

Address corrections: 17
Exported: ../data/cleaned/replies/d1_address_corrections.csv


,address_clean,corrected_address_clean
0,4102 8007 Ziata Ct 134 8,8007 Zlata Ct
1,22796 4622 FlorinR '245,4622 Florin Rd #6
2,3384 5200 Stockton. '25,5200 Stockton Blvd #130
3,T1608 4317 54th St 'ids,4317 54th St
4,4567746750 Ave '20501,7467 50 Ave
5,46333932 33thSt '/ds!,3932 39th St
6,682 7501StstAve ' el,7501 51st Ave
7,22956 637 Fairgroun '24 5,637 Fairgrounds Dr #3
8,12155 584125 Ave [ids,5841 25 Ave
9,13530 3908 42nd Av '24 5,3908 42nd Ave #E


## 16.17 Apply Address Corrections

Address corrections identified during manual QA are applied before rebuilding the standardized address fields.

Applying corrections prior to address parsing ensures corrected reply records follow the same automated processing pipeline as all other records.

In [67]:
corrections = pd.read_csv(
    REPLIES_DIR / "d1_address_corrections.csv"
)

correction_map = dict(
    zip(
        corrections["address_clean"],
        corrections["corrected_address_clean"]
    )
)

reply_tracker["address_clean"] = (
    reply_tracker["address_clean"]
    .replace(correction_map)
)

print(f"Corrections applied: {len(correction_map):,}")

display(
    reply_tracker[
        [
            "campaign",
            "address_clean",
            "reply_text"
        ]
    ].head(20)
)

Corrections applied: 17


,campaign,address_clean,reply_text
0,D1_A,130 7350 Alden Way,| think you moved 1 mile out my r...
1,D1_A,4080 6425 Somis,Yes
2,D1_A,13755 2174 55 Ave,YES
3,D1_A,10189 5505 Sky Parkway &,Yes
4,D1_A,20945 6618 Sunriver Dr,Yes
5,D1_A,867 5642 Odea DrApt3,Yes
6,D1_A,7200 5923 69th St,Yes
7,D1_A,6246 6311 Sampson Blvd &,Yes
8,D1_A,4018 7831 Lemon Hill Ave &,Yes!
9,D1_A,21209 7732 40 Ave,YES
